# Machine Learning en Python : projet de classification complet

**Question :** à la fin du premier trimestre, peut-on repérer les élèves qui risquent
d'échouer en mathématiques, pour leur proposer un soutien à temps ?

On va suivre, dans l'ordre, les étapes de la formation :

1. **Collecte** — charger les données
2. **Exploration** — premier examen
3. **Nettoyage** — un jeu sans valeur manquante est-il propre ?
4. **Exploration et prétraitement** — date de prédiction, découpage, préparation
5. **Modélisation** — entraîner une régression logistique
6. **Interprétation** — évaluer, régler le seuil, comparer

*Outil pédagogique et hypothétique : il ne sert à aucun repérage réel d'élèves.*

*Données : Student Performance, UCI Machine Learning Repository — P. Cortez et A. Silva,
« Using Data Mining to Predict Secondary School Student Performance », FUBUTEC 2008 —
https://doi.org/10.24432/C5TG7T — licence CC BY 4.0.*

## 1. Collecte — charger les données

**395 élèves** de deux lycées publics portugais, année scolaire 2005-2006.
**Une ligne représente un élève** et ses résultats en mathématiques sur l'année.

Notre objectif : prédire une **catégorie**, réussite ou échec, et non une quantité.
C'est donc un problème de **classification**.

In [ ]:
import pandas as pd

URL = ("https://raw.githubusercontent.com/formation-ia-data/"
       "classification-eleves-python/v1/data/student-mat.csv")
df = pd.read_csv(URL)
df.shape

In [ ]:
df.head(2)

**Une seule colonne**, dont le nom est une longue chaîne remplie de `;`.
Le fichier utilise le **point-virgule** comme séparateur : c'est l'usage européen, parce que
la virgule y sert de séparateur décimal. Il faut le dire à pandas.

In [ ]:
df = pd.read_csv(URL, sep=";")
df.shape

## 2. Exploration — premier examen

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
df.isna().sum().sum()

33 colonnes, c'est trop pour tout lire. Une vue réduite :

In [ ]:
df[["school", "age", "studytime", "failures", "G1", "G2", "G3"]].head()

`G1`, `G2` et `G3` sont les notes des trois périodes de l'année, sur 20.
`G3` est la note finale.

### La variable cible

Un élève est en **échec** si sa note finale est inférieure à 10.

In [ ]:
df["echec"] = (df["G3"] < 10).astype(int)
df["echec"].value_counts()

In [ ]:
df["echec"].value_counts(normalize=True).round(3)

## 3. Nettoyage

Aucune valeur manquante. Le jeu est-il propre pour autant ?

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df["G3"], bins=range(0, 22), edgecolor="white")
plt.xlabel("Note finale G3")
plt.ylabel("Nombre d'élèves")
plt.show()

Une barre isolée à 0, et **personne entre 1 et 3**. Regardons ces élèves.

In [ ]:
zeros = df[df["G3"] == 0]
len(zeros)

In [ ]:
zeros[["G1", "G2", "G3", "absences"]].describe()

Comparons avec les autres élèves en échec, ceux qui ont une note finale de 1 à 9 :

In [ ]:
autres_echecs = df[(df["G3"] >= 1) & (df["G3"] <= 9)]

print("Note finale à 0      :", len(zeros), "élèves,",
      round(zeros["absences"].mean(), 1), "absences en moyenne,",
      f"{(zeros['absences'] == 0).mean():.0%} à zéro absence")
print("Note finale de 1 à 9 :", len(autres_echecs), "élèves,",
      round(autres_echecs["absences"].mean(), 1), "absences en moyenne,",
      f"{(autres_echecs['absences'] == 0).mean():.0%} à zéro absence")

Les 38 élèves à 0 ont **tous** zéro absence : un profil très atypique.
L'hypothèse « examen non passé » est **plausible, mais non démontrée** : la documentation
annonce des notes de 0 à 20 et aucune valeur manquante.

Les retirer modifierait fortement la population étudiée, et changerait le repère du
modèle naïf « tout le monde réussit » :

In [ ]:
sans_zeros = df[df["G3"] > 0]
print("Modèle naïf, avec les 38 :", f"{1 - df['echec'].mean():.1%}")
print("Modèle naïf, sans les 38 :", f"{1 - sans_zeros['echec'].mean():.1%}")

**Décision :** faute de savoir ce que signifie exactement ce zéro, on **garde** ces
38 élèves. On vérifiera à la fin si les conclusions tiennent sans eux.

Une anomalie n'est pas automatiquement une valeur manquante.

*Optionnel.* La documentation décrit `failures` comme « n si 1 ≤ n < 3, sinon 4 ».
Le fichier, lui :

In [ ]:
df["failures"].value_counts().sort_index()

Aucun 4, et un maximum à 3 : la documentation et le fichier ne concordent pas.
On le signale, sans l'interpréter.

## 4. Exploration et prétraitement

### 4a. La date de prédiction

On **simule** une prédiction **à la fin du premier trimestre**, quand on peut encore aider.
Cette date délimite les colonnes qu'on a le droit d'utiliser.

| Disponible à la date | Postérieur | Disponibilité inconnue (hypothèse) |
|---|---|---|
| `G1` | `G2`, `G3` | `school`, `age`, `studytime`, `failures`, `internet` : questionnaire non daté |
| | | `absences` : période de comptage inconnue → **retirée** |

On **suppose** que les réponses au questionnaire étaient connues à cette date :
le dataset ne permet pas de le vérifier.

On écarte aussi, **par choix** :
- `schoolsup` (soutien scolaire) : il a pu être mis en place *à cause* d'un mauvais
  premier trimestre, et un outil qui propose du soutien ne doit pas le prendre en entrée ;
- `higher` (envie de faire des études supérieures) : un élève ne doit pas recevoir moins
  d'attention pour cette raison ;
- les variables personnelles et familiales (sexe, parents, alcool, vie amoureuse…).

Les six variables d'entrée sont fixées **avant** toute exploration de leur association
avec la cible.

Ce choix ne rend pas le modèle neutre : l'établissement ou les échecs passés peuvent agir
comme des **indicateurs indirects** et reproduire des inégalités présentes dans les données.

In [ ]:
colonnes = ["school", "age", "studytime", "failures", "internet", "G1"]
X = df[colonnes].copy()
y = df["echec"]

# Encodage binaire, comme l'exemple du fumeur dans le cours.
# Il n'apprend rien des données : on peut le faire avant le découpage.
X["internet"] = (X["internet"] == "yes").astype(int)
X["school"] = (X["school"] == "MS").astype(int)
X.head()

`studytime` est **déjà** un encodage ordinal : 1 = moins de 2 h par semaine,
2 = 2 à 5 h, 3 = 5 à 10 h, 4 = plus de 10 h.

### 4b. Découper, tout de suite

Trois ensembles, comme dans le cours : **apprentissage** (60 %), **validation** (20 %),
**test** (20 %). `stratify` garde la même proportion d'échecs dans chacun.

À partir d'ici, **tout calcul qui regarde la cible se fait sur l'apprentissage**.
Le test ne sert à aucun réglage : il ne sert qu'à l'évaluation finale.

In [ ]:
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp
)

In [ ]:
for nom, cible in [("apprentissage", y_train), ("validation", y_val), ("test", y_test)]:
    nb_zeros = (df.loc[cible.index, "G3"] == 0).sum()
    print(f"{nom:14s}: {len(cible)} élèves, {cible.sum()} échecs ({cible.mean():.1%}),"
          f" dont {nb_zeros} à 0")

### 4c. Explorer, sur l'apprentissage seulement

In [ ]:
appr = df.loc[X_train.index]
appr.groupby("failures")["echec"].agg(["mean", "count"]).round(2)

In [ ]:
appr[["G1", "G2", "G3"]].corr()["G3"].round(3)

`G2` est encore plus liée à la note finale que `G1`. Mais à la fin du premier trimestre,
**on ne la connaît pas** : l'utiliser serait une **fuite de cible**.

In [ ]:
plt.scatter(appr["G1"], appr["G3"], alpha=0.4)
plt.axhline(10, color="grey", linestyle="--")
plt.xlabel("Note du premier trimestre G1")
plt.ylabel("Note finale G3")
plt.show()

### 4d. Standardiser

Contrairement à l'encodage, la standardisation **apprend** une moyenne et un écart-type :
on l'ajuste donc **sur l'apprentissage seulement**, puis on l'applique aux autres ensembles.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

## 5. Modélisation — entraîner une régression logistique

Classe positive = **échec** (1) : « positif » désigne ce qu'on cherche à repérer,
pas une bonne nouvelle.

In [ ]:
from sklearn.linear_model import LogisticRegression

modele = LogisticRegression()
modele.fit(X_train_s, y_train)

proba_val = modele.predict_proba(X_val_s)[:, 1]   # probabilité d'échec estimée
pred_val  = modele.predict(X_val_s)               # seuil de 0,5

`predict_proba` applique la **fonction sigmoïde** : elle transforme un score en
**probabilité estimée par le modèle**. Quelques élèves de la validation :

In [ ]:
apercu = df.loc[X_val.index, ["G1", "failures", "studytime", "echec"]].copy()
apercu["proba_echec"] = proba_val.round(2)
apercu.head(8)

`predict` déclare « échec » dès que cette probabilité atteint **0,5**.
Ce seuil est un **choix**, pas une loi.

Les coefficients : on lit leur **signe**, pas leur valeur exacte.

In [ ]:
pd.Series(modele.coef_[0], index=colonnes).round(2).sort_values()

## 6. Interprétation — évaluer, régler le seuil, comparer

### 6a. Sur la validation

Une petite fonction pour ne pas répéter les mêmes lignes à chaque modèle.
Elle traduit les mesures **en élèves**.

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

def bilan(nom, y_vrai, y_pred):
    vn, fp, fn, vp = confusion_matrix(y_vrai, y_pred, labels=[0, 1]).ravel()
    print(nom)
    print(f"  taux de bonne prédiction : {accuracy_score(y_vrai, y_pred):.1%}")
    print(f"  rappel (échec)           : {vp} élèves repérés sur {vp + fn} en échec"
          f" ({recall_score(y_vrai, y_pred, zero_division=0):.1%})")
    if vp + fp > 0:
        print(f"  précision (échec)        : {vp} vrais échecs sur {vp + fp} alertes"
              f" ({precision_score(y_vrai, y_pred):.1%})")
    else:
        print("  précision (échec)        : non définie, aucune alerte (0 par convention)")
    print(f"  score F1 (échec)         : {f1_score(y_vrai, y_pred, zero_division=0):.3f}")
    print(f"  fausses alertes : {fp} · élèves manqués : {fn}")

In [ ]:
pred_naif_val = np.zeros(len(y_val), dtype=int)     # « tout le monde réussit »

bilan("Modèle naïf", y_val, pred_naif_val)
bilan("Régression logistique, seuil 0,5", y_val, pred_val)

Le modèle naïf a un taux de bonne prédiction honorable… et **ne repère aucun élève en échec**.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val, pred_val, display_labels=["réussite (0)", "échec (1)"], colorbar=False
)
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.title("Régression logistique, seuil 0,5 — validation")
plt.show()

### Le modèle « tricheur » : avec `G2`

Même chose, en ajoutant la note du deuxième trimestre :

In [ ]:
X_g2 = X.copy()
X_g2["G2"] = df["G2"]

scaler_g2 = StandardScaler().fit(X_g2.loc[X_train.index])
modele_g2 = LogisticRegression().fit(scaler_g2.transform(X_g2.loc[X_train.index]), y_train)
pred_g2_val = modele_g2.predict(scaler_g2.transform(X_g2.loc[X_val.index]))

bilan("Régression logistique AVEC G2, seuil 0,5", y_val, pred_g2_val)

Quel que soit ce score, ce modèle est **inutile** : quand on connaît `G2`, l'année est
presque finie.
**Un score élevé n'est pas une preuve d'utilité.**

### 6b. Régler le seuil, sur la validation

**Objectif fixé avant le calcul :** repérer au moins 8 élèves en échec sur 10 (rappel ≥ 80 %),
avec le moins de fausses alertes possible (la meilleure précision).
À précision égale, on prend le rappel le plus élevé, puis le seuil qui déclenche le moins
d'alertes.

Ce rappel de 80 % est un **objectif de réglage sur la validation**, pas une performance
promise sur le test.

In [ ]:
def choisir_seuil(y_vrai, predictions_par_seuil):
    candidats = []
    for seuil, pred in predictions_par_seuil.items():
        rappel = recall_score(y_vrai, pred)
        if rappel >= 0.80:
            precision = precision_score(y_vrai, pred, zero_division=0)
            candidats.append((precision, rappel, -pred.sum(), seuil))
    return max(candidats)[3]

On applique **la même règle** à deux concurrents :
- la régression logistique, avec un seuil sur la probabilité estimée ;
- la **note `G1` seule** : « alerte si `G1` < c ». `G1` est déjà un score : on peut aussi
  déplacer son seuil.

In [ ]:
from sklearn.metrics import precision_recall_curve

_, _, seuils = precision_recall_curve(y_val, proba_val)
seuil_reg = choisir_seuil(
    y_val, {s: (proba_val >= s).astype(int) for s in seuils}
)

notes = sorted(X_val["G1"].unique())
c_g1 = choisir_seuil(
    y_val, {c: (X_val["G1"] < c).astype(int).to_numpy() for c in notes}
)

print("Seuil retenu pour la régression :", round(seuil_reg, 3))
print("Seuil retenu pour G1            : alerte si G1 <", c_g1)

In [ ]:
bilan(f"Régression logistique, seuil {seuil_reg:.2f}", y_val,
      (proba_val >= seuil_reg).astype(int))
bilan(f"Note G1 seule, alerte si G1 < {c_g1}", y_val,
      (X_val["G1"] < c_g1).astype(int))

Abaisser le seuil repère plus d'élèves, au prix de plus de fausses alertes.
Une fausse alerte a un coût : du temps d'enseignant, et un élève étiqueté « à risque » à tort.

### 6c. L'évaluation finale, sur le test

Les seuils sont **figés**. On ne les déplace plus.

In [ ]:
proba_test   = modele.predict_proba(X_test_s)[:, 1]
pred_naif    = np.zeros(len(y_test), dtype=int)
pred_g1      = (X_test["G1"] < c_g1).astype(int)
pred_reg     = (proba_test >= seuil_reg).astype(int)

bilan("Modèle naïf", y_test, pred_naif)
bilan(f"Note G1 seule, alerte si G1 < {c_g1}", y_test, pred_g1)
bilan(f"Régression logistique, seuil {seuil_reg:.2f}", y_test, pred_reg)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, pred, titre in [(axes[0], pred_g1, f"G1 < {c_g1}"),
                        (axes[1], pred_reg, f"Régression, seuil {seuil_reg:.2f}".replace(".", ","))]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["réussite (0)", "échec (1)"],
        ax=ax, colorbar=False
    )
    ax.set_xlabel("Classe prédite")
    ax.set_ylabel("Classe réelle")
    ax.set_title(titre + " — test")
plt.tight_layout()
plt.show()

Le test ne compte qu'une **vingtaine d'échecs** : un seul élève fait bouger le rappel de
près de 4 points. Si les deux méthodes sont proches, ou donnent des compromis différents,
**on ne désigne pas de vainqueur**.

### 6d. Et sans les 38 élèves à 0 ?

On garde **les mêmes ensembles**, et on retire les élèves à 0 à l'intérieur de chacun.
Puis on refait tout : standardisation, modèle, réglage des seuils sur la validation,
évaluation sur le test.

In [ ]:
def sans_les_zeros(X_, y_):
    garder = (df.loc[X_.index, "G3"] > 0).to_numpy()
    return X_[garder], y_[garder]

X_train_f, y_train_f = sans_les_zeros(X_train, y_train)
X_val_f,   y_val_f   = sans_les_zeros(X_val, y_val)
X_test_f,  y_test_f  = sans_les_zeros(X_test, y_test)

scaler_f = StandardScaler().fit(X_train_f)
modele_f = LogisticRegression().fit(scaler_f.transform(X_train_f), y_train_f)

proba_val_f = modele_f.predict_proba(scaler_f.transform(X_val_f))[:, 1]
_, _, seuils_f = precision_recall_curve(y_val_f, proba_val_f)
seuil_reg_f = choisir_seuil(
    y_val_f, {s: (proba_val_f >= s).astype(int) for s in seuils_f}
)
c_g1_f = choisir_seuil(
    y_val_f, {c: (X_val_f["G1"] < c).astype(int).to_numpy()
              for c in sorted(X_val_f["G1"].unique())}
)

proba_test_f = modele_f.predict_proba(scaler_f.transform(X_test_f))[:, 1]
bilan(f"SANS LES ZÉROS — G1 < {c_g1_f}", y_test_f, (X_test_f["G1"] < c_g1_f).astype(int))
bilan(f"SANS LES ZÉROS — régression, seuil {seuil_reg_f:.2f}", y_test_f,
      (proba_test_f >= seuil_reg_f).astype(int))

Le coefficient de `G1`, **ramené à un point de note** (le coefficient standardisé change
d'unité quand l'écart-type change) :

In [ ]:
i = colonnes.index("G1")
print("Avec les 38 :", round(modele.coef_[0][i] / scaler.scale_[i], 3), "par point de G1")
print("Sans les 38 :", round(modele_f.coef_[0][i] / scaler_f.scale_[i], 3), "par point de G1")

---
### Complément — la variabilité des mesures

Une **validation croisée descriptive**, sur l'apprentissage et la validation réunis
(**jamais le test**), au seuil par défaut de 0,5. Elle ne choisit ni seuil ni modèle, et ne
change pas la conclusion : elle montre seulement combien les mesures varient d'un découpage
à l'autre. Le `make_pipeline` refait la standardisation à l'intérieur de chaque pli, pour
éviter toute fuite.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline

X_av = pd.concat([X_train, X_val])
y_av = pd.concat([y_train, y_val])

resultats_cv = cross_validate(
    make_pipeline(StandardScaler(), LogisticRegression()),
    X_av, y_av,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=["accuracy", "precision", "recall"],
)
pd.DataFrame(resultats_cv)[["test_accuracy", "test_precision", "test_recall"]] \
    .describe().loc[["mean", "min", "max"]].round(3)

---
### Ce que ce modèle est, et ce qu'il n'est pas

Un **signal gradué**, calculé tôt, qu'un enseignant examinerait pour proposer un soutien.
Il ne décide pas à sa place. Il décrit des **associations** observées dans deux lycées
portugais, sur une année : pas des causes, et rien qui vaille ailleurs sans vérification.

---
### Versions utilisées pour la vidéo

Si le notebook ne fonctionne plus un jour, cette cellule indique avec quelles versions
la vidéo a été tournée.

In [ ]:
import pandas, sklearn, matplotlib, sys
print(sys.version)
print("pandas", pandas.__version__, "| sklearn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__)